# Distributed PyTorch training with Ray Train — official Ray Train DLC

Distributed data-parallel PyTorch training with `ray.train.torch.TorchTrainer` (ResNet on CIFAR-10) on AWS's official **Ray Train Deep Learning Container** (`train-ml-cuda-v1.1`) — see [`../README.md`](../README.md) for what's in the image and how the direct image pull was confirmed. A single multi-GPU instance, using one unmodified `launcher.py`.

***

## Prerequisites

In [ ]:
%pip install -r ./scripts/requirements.txt --upgrade

In [ ]:
# Copy the shared Ray launcher script into ./scripts (not committed here).
%cp ../../../scripts/launcher.py ./scripts/

***

# Step 1 - Session and role

In [ ]:
from sagemaker.core.helper.session_helper import get_execution_role, Session

In [ ]:
sagemaker_session = Session()

bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix
role = get_execution_role()

# Step 2 - Resolve the Ray Train DLC image

AWS also publishes this image directly in a private, per-region ECR account under repository `ray` — the same distribution mechanism every other DLC (`pytorch-training`, `huggingface-training`, ...) already uses, with the same broad cross-account pull permissions. `sagemaker:CreateTrainingJob` pulls from there directly; `ray_dlc_image.get_ray_train_dlc_image_uri()` just resolves the right URI for your region — see [`../README.md`](../README.md) for how this was confirmed.

In [ ]:
import sys
sys.path.insert(0, "..")
from ray_dlc_image import get_ray_train_dlc_image_uri

image_uri = get_ray_train_dlc_image_uri(sagemaker_session.boto_session.region_name)
image_uri

# Step 3 - Launch the training job (homogeneous cluster)

A single `ml.g5.12xlarge` (4x A10G). The head node is also a worker, and `train.py` auto-sizes `num_workers` to the 4 GPUs it finds.

In [ ]:
! pygmentize ./scripts/train.py

In [ ]:
from sagemaker.train.configs import (
    CheckpointConfig,
    Compute,
    OutputDataConfig,
    RemoteDebugConfig,
    SourceCode,
    StoppingCondition,
)
from sagemaker.train.model_trainer import ModelTrainer

args = [
    "-e",
    "train.py",
    "--epochs",
    "20",
    "--learning_rate",
    "0.001",
    "--batch_size",
    "128",
]

instance_type = "ml.g5.12xlarge"
instance_count = 1

source_code = SourceCode(
    source_dir="./scripts",
    requirements="requirements.txt",
    command=f"python launcher.py {' '.join(args)}",
)

compute_configs = Compute(
    instance_type=instance_type,
    instance_count=instance_count,
    keep_alive_period_in_seconds=0,
)

job_name = "train-ray-train-cifar10"
if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{job_name}"
else:
    output_path = f"s3://{bucket_name}/{job_name}"

model_trainer = ModelTrainer(
    training_image=image_uri,
    source_code=source_code,
    base_job_name=job_name,
    compute=compute_configs,
    stopping_condition=StoppingCondition(max_runtime_in_seconds=18000),
    output_data_config=OutputDataConfig(
        s3_output_path=output_path, compression_type="NONE"
    ),
    checkpoint_config=CheckpointConfig(
        s3_uri=output_path + "/checkpoints", local_path="/opt/ml/checkpoints"
    ),
    environment={},
    role=role,
).with_remote_debug_config(RemoteDebugConfig(enable_remote_debug=True))

In [ ]:
# Start the training job.
model_trainer.train(wait=False)

The rank-0 worker writes checkpoints to `/opt/ml/checkpoints`, which SageMaker syncs to `s3://.../checkpoints` via the `CheckpointConfig` above.

**Note:** `train.py`'s dataset download (torchvision's default CIFAR-10 mirror) can be slow depending on network path — give the job enough time (or stage the dataset via an S3 input channel instead) to reliably reach a completed epoch and checkpoint. See [`../README.md`](../README.md).